In [ ]:
import os
from pathlib import Path
import json
import torch
from faster_whisper import WhisperModel
from pydub import AudioSegment
from nemo.collections.asr.models import SortformerEncLabelModel, EncDecSpeakerLabelModel
from audio_utils import trim_wav
import numpy as np
import librosa


In [3]:
# Create 2 segments
tld = Path(r"C:\Users\Somlab\Downloads")
audio_path = tld / "audio1365983309.16k.wav"
# trim_wav(audio_path, tld / "segment_1.wav", 0, 90)
# trim_wav(audio_path, tld / "segment_2.wav", 100, 190)

trim_wav(audio_path, tld / "audio1365983309_trunc.wav", 0, 600)

Successfully trimmed C:\Users\Somlab\Downloads\audio1365983309.16k.wav to C:\Users\Somlab\Downloads\audio1365983309_trunc.wav.


In [ ]:
diar_model = SortformerEncLabelModel.from_pretrained("nvidia/diar_sortformer_4spk-v1")
diar_model.eval()
manifest_path = audio_path.with_suffix(".json")
with open(manifest_path, "w", encoding="utf-8") as f:
    f.write(json.dumps({"audio_filepath": str(audio_path),
                        "offset": 0,
                        "duration": 90,
                        "label": "infer",
                        "text": "-"}) + "\n")

# Run the diarization
prediction = diar_model.diarize(audio=[str(manifest_path)], batch_size=1)


In [ ]:
process_prediction(prediction[0])

In [ ]:
def extract_titanet_embedding(model, audio_path, start_time, end_time):
    # 1. Identify which device the model is currently using (CPU or CUDA)
    device = next(model.parameters()).device
    
    # 2. Load audio slice
    y, sr = librosa.load(audio_path, sr=16000, offset=start_time, duration=(end_time - start_time))
    
    # 3. Create tensors and explicitly send them to the correct device
    audio_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0).to(device)
    
    # Length must be an integer tensor representing the number of samples
    audio_len = torch.tensor([audio_tensor.shape[1]], dtype=torch.int32).to(device)
    
    # 4. Extract embedding vector on the correct device
    with torch.no_grad():
        _, embedding = model.forward(input_signal=audio_tensor, input_signal_length=audio_len)
    
    # 5. Bring back to CPU for numpy operations
    embedding = embedding.squeeze(0).cpu().numpy()
    return embedding / np.linalg.norm(embedding)

In [ ]:
titanet = EncDecSpeakerLabelModel.from_pretrained(model_name="titanet_small").eval()
emb_1 = extract_titanet_embedding(titanet, audio_path, 0, 20)
emb_2 = extract_titanet_embedding(titanet, audio_path, 21, 70)
emb_3 = extract_titanet_embedding(titanet, audio_path, 72, 85)
emb_4 = extract_titanet_embedding(titanet, audio_path, 86, 212) 

In [ ]:
np.dot(emb_2, emb_4)
